In [ ]:
# ============ CELL 1: SETUP (GPU check, installs, paths) ============
import os, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=False)

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU -> Runtime > Change runtime type > T4 GPU")

USE_DRIVE = False   # True = mount Google Drive so results survive a disconnect (recommended)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/uap_work"
else:
    WORK_DIR = "/content/work"
os.makedirs(WORK_DIR, exist_ok=True)
print("WORK_DIR =", WORK_DIR)


torch 2.11.0+cu128 | cuda: True | Tesla T4
WORK_DIR = /content/work


In [ ]:
# ============ CELL 2: DATA + EXACT BASELINE (text TF-IDF LR + ID prior), cached ============
import os, re, json, time, math, gc, warnings
import numpy as np, pandas as pd, scipy.sparse as sp
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
warnings.filterwarnings("ignore")

SEED, N_SPLITS, K_NEIGHBORS, ID_EPS = 42, 5, 20, 1e-6
WORK = Path(WORK_DIR)

def find_data_dir():
    for p in ["/content", "/content/drive/MyDrive", ".", "/mnt/user-data/uploads"]:
        if (Path(p) / "train.csv").exists() and (Path(p) / "test.csv").exists():
            return Path(p)
    raise FileNotFoundError("train.csv / test.csv not found - upload them to /content")
DATA_DIR = find_data_dir()

URL_RE = re.compile(r"(?:https?://|www\.)\S+", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
WS_RE = re.compile(r"\s+", flags=re.UNICODE)
def clean_text(t):
    s = "" if t is None else str(t)
    s = URL_RE.sub(" URL ", s); s = MENTION_RE.sub(" USER ", s)
    return WS_RE.sub(" ", s).strip()

train = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8")
test = pd.read_csv(DATA_DIR / "test.csv", encoding="utf-8")
assert list(train.columns) == ["id", "text", "y"] and list(test.columns) == ["id", "text"]
for df in (train, test):
    df["text"] = df["text"].astype("string").fillna("").astype(str)
clean_texts = train["text"].map(clean_text).to_numpy()
test_clean = test["text"].map(clean_text).to_numpy()
y = train["y"].to_numpy(dtype=int)
ids = train["id"].to_numpy(dtype=np.int64)
test_ids = test["id"].to_numpy(dtype=np.int64)
assert len(set(ids) & set(test_ids)) == 0
print("train", train.shape, "test", test.shape, "classes", np.bincount(y))

# ---- canonical folds: StratifiedGroupKFold(5, shuffle, 42), group = cleaned+lowercased text ----
def build_folds():
    keys = pd.Index([str(t).lower().strip() for t in clean_texts], dtype="object")
    groups = pd.factorize(keys, sort=False)[0]
    fold_ids = np.full(len(y), -1, dtype=int)
    for k, (_, va) in enumerate(StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=SEED)
                                .split(np.zeros(len(y)), y, groups)):
        fold_ids[va] = k
    cross = pd.DataFrame({"g": groups, "f": fold_ids}).groupby("g")["f"].nunique()
    assert (fold_ids >= 0).all() and int((cross > 1).sum()) == 0, "leak: group spans folds"
    return fold_ids
fold_ids = build_folds()
def fold_split(k): return np.flatnonzero(fold_ids != k), np.flatnonzero(fold_ids == k)
print("fold sizes", np.bincount(fold_ids))

# ---- ID-locality prior (verbatim baseline logic; loo=True lets TRAIN rows get a self-excluded prior) ----
def id_prior(query_ids, pool_ids, pool_labels, k=K_NEIGHBORS, eps=ID_EPS, loo=False):
    q = np.asarray(query_ids, np.int64).ravel(); t = np.asarray(pool_ids, np.int64).ravel()
    yy = np.asarray(pool_labels, np.int64).ravel()
    order = np.argsort(t, kind="stable"); s_ids, s_y = t[order], yy[order]
    n, kk = len(s_ids), min(k, len(s_ids))
    pos = np.searchsorted(s_ids, q, side="left")
    cand = pos[:, None] + np.arange(-k, k + 1)[None, :]
    valid = (cand >= 0) & (cand < n); cc = np.clip(cand, 0, n - 1)
    d = np.abs(s_ids[cc] - q[:, None]).astype(np.float64)
    d = np.where(valid & (s_ids[cc] != q[:, None]), d, np.inf)      # never match itself
    sel = np.argsort(d, axis=1, kind="stable")[:, :kk]
    dsel = np.take_along_axis(d, sel, axis=1)
    ysel = s_y[cc[np.arange(len(q))[:, None], sel]]
    good = np.isfinite(dsel)
    w = np.where(good, 1.0 / (np.where(good, dsel, 0.0) + eps), 0.0)
    P = np.stack([(w * (ysel == c)).sum(axis=1) for c in range(3)], axis=1)
    tot = P.sum(axis=1, keepdims=True)
    P = P / np.where(tot > 0, tot, 1.0); P[tot.ravel() <= 0] = 1 / 3
    if not loo:
        assert int(np.isin(q, s_ids).sum()) == 0, "leak: query ids present in prior pool"
    return P

def mf1(yt, pred):   # fast macro-F1 (verified == sklearn)
    cm = np.bincount(np.asarray(yt) * 3 + np.asarray(pred), minlength=9).reshape(3, 3)
    tp = np.diag(cm).astype(float)
    p = tp / np.maximum(cm.sum(0), 1); r = tp / np.maximum(cm.sum(1), 1)
    return float((2 * p * r / np.maximum(p + r, 1e-12)).mean())
def mf1_proba(yt, P): return mf1(yt, np.asarray(P).argmax(1))
def per_class(yt, P):
    cm = np.bincount(np.asarray(yt) * 3 + np.asarray(P).argmax(1), minlength=9).reshape(3, 3)
    tp = np.diag(cm).astype(float); p = tp / np.maximum(cm.sum(0), 1); r = tp / np.maximum(cm.sum(1), 1)
    return [round(float(v), 4) for v in 2 * p * r / np.maximum(p + r, 1e-12)]

# ---- baseline sources (cached) ----
BASE = WORK / "base_sources.npz"
if BASE.exists():
    z = np.load(BASE); oof_text, oof_id, test_text, test_id = z["oof_text"], z["oof_id"], z["test_text"], z["test_id"]
    print("loaded cached baseline sources")
else:
    t0 = time.time()
    WORD_KW = dict(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=60000, sublinear_tf=True, dtype=np.float32)
    CHAR_KW = dict(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=60000, sublinear_tf=True, dtype=np.float32)
    LR_KW = dict(solver="lbfgs", C=4.0, class_weight="balanced", max_iter=2000, random_state=SEED)
    def feats(fit_texts, *apply):
        w = TfidfVectorizer(**WORD_KW).fit(fit_texts); c = TfidfVectorizer(**CHAR_KW).fit(fit_texts)
        return [sp.hstack([w.transform(a), c.transform(a)], format="csr").astype(np.float32) for a in apply]
    oof_text = np.zeros((len(y), 3)); oof_id = np.zeros((len(y), 3))
    for k in range(N_SPLITS):
        tr, va = fold_split(k)
        Xtr, Xva = feats(clean_texts[tr], clean_texts[tr], clean_texts[va])
        oof_text[va] = LogisticRegression(**LR_KW).fit(Xtr, y[tr]).predict_proba(Xva)
        oof_id[va] = id_prior(ids[va], ids[tr], y[tr])
        print(f"baseline fold {k} done {time.time()-t0:.0f}s", flush=True)
    Xtr, Xte = feats(clean_texts, clean_texts, test_clean)
    test_text = LogisticRegression(**LR_KW).fit(Xtr, y).predict_proba(Xte)
    test_id = id_prior(test_ids, ids, y)
    np.savez(BASE, oof_text=oof_text, oof_id=oof_id, test_text=test_text, test_id=test_id)

print(f"text-only  OOF macroF1 = {mf1_proba(y, oof_text):.6f}   (team ref 0.512943)")
print(f"id-only    OOF macroF1 = {mf1_proba(y, oof_id):.6f}   (team ref 0.614135)")
bl = 0.4 * oof_text + 0.6 * oof_id
print(f"text+id@.4 OOF macroF1 = {mf1_proba(y, bl):.6f}   (team ref 0.639579)")


train (47817, 3) test (11955, 2) classes [23756 18163  5898]
fold sizes [9543 9546 9571 9581 9576]
baseline fold 0 done 87s
baseline fold 1 done 143s
baseline fold 2 done 197s
baseline fold 3 done 248s
baseline fold 4 done 302s
text-only  OOF macroF1 = 0.515571   (team ref 0.512943)
id-only    OOF macroF1 = 0.613631   (team ref 0.614135)
text+id@.4 OOF macroF1 = 0.640821   (team ref 0.639579)


In [ ]:
# ============ CELL 3: TRANSFORMER FINE-TUNING (5 grouped folds, resumable) ============
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CUDA = DEVICE == "cuda"
RUN_DIR = WORK / "runs"; RUN_DIR.mkdir(exist_ok=True)

def get_tokenizer(name): return AutoTokenizer.from_pretrained(name)
def get_backbone(name):  return AutoModel.from_pretrained(name)

class Clf(nn.Module):
    """[CLS] -> dropout -> linear(3).  If use_prior: logits += Linear(3,3)(log ID-prior) (zero-init, learned jointly)."""
    def __init__(self, backbone, hidden, use_prior, dropout=0.1):
        super().__init__()
        self.backbone, self.drop, self.head, self.use_prior = backbone, nn.Dropout(dropout), nn.Linear(hidden, 3), use_prior
        if use_prior:
            self.prior_lin = nn.Linear(3, 3)
            nn.init.zeros_(self.prior_lin.weight); nn.init.zeros_(self.prior_lin.bias)
    def forward(self, input_ids, attention_mask, prior=None):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        logits = self.head(self.drop(h))
        if self.use_prior:
            logits = logits + self.prior_lin(torch.log(prior.clamp_min(1e-3)))
        return logits

def make_batch(tok_ids, rows, pad_id):
    L = max(len(tok_ids[i]) for i in rows)
    inp = torch.full((len(rows), L), pad_id, dtype=torch.long); att = torch.zeros((len(rows), L), dtype=torch.long)
    for r, i in enumerate(rows):
        s = tok_ids[i]; inp[r, :len(s)] = torch.tensor(s, dtype=torch.long); att[r, :len(s)] = 1
    return inp.to(DEVICE), att.to(DEVICE)

@torch.no_grad()
def predict(model, tok_ids, rows, prior, pad_id, bs=128):
    model.eval()
    rows = np.asarray(rows); order = np.argsort([len(tok_ids[i]) for i in rows])
    out = np.zeros((len(rows), 3), dtype=np.float32)
    for s in range(0, len(rows), bs):
        pos = order[s:s + bs]
        inp, att = make_batch(tok_ids, rows[pos], pad_id)
        pr = torch.tensor(prior[pos], dtype=torch.float32, device=DEVICE) if prior is not None else None
        with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
            logits = model(inp, att, pr)
        out[pos] = torch.softmax(logits.float(), -1).cpu().numpy()
    return out

def run_fold(cfg, k, tok_train, tok_test, pad_id, smoke=False):
    tr, va = fold_split(k)
    rng = np.random.RandomState(SEED + k)
    if smoke:
        tr, va = rng.choice(tr, 1500, replace=False), rng.choice(va, 400, replace=False)
    use_prior = cfg["use_prior"]
    # leak-free priors: train rows -> leave-one-out over fold-train pool; val rows -> fold-train pool; test -> full train pool
    p_tr = id_prior(ids[tr], ids[tr], y[tr], loo=True) if use_prior else None
    p_va = id_prior(ids[va], ids[tr], y[tr]) if use_prior else None
    p_te = id_prior(test_ids, ids, y) if use_prior else None

    torch.manual_seed(SEED + k); torch.cuda.manual_seed_all(SEED + k)
    backbone = get_backbone(cfg["model"])
    model = Clf(backbone, backbone.config.hidden_size, use_prior).to(DEVICE)
    bb = [p for n, p in model.named_parameters() if n.startswith("backbone.")]
    hd = [p for n, p in model.named_parameters() if n.startswith("head.")]
    groups = [dict(params=bb, lr=cfg["lr"], weight_decay=0.01), dict(params=hd, lr=cfg["lr"] * 5, weight_decay=0.01)]
    if use_prior:
        groups.append(dict(params=list(model.prior_lin.parameters()), lr=5e-3, weight_decay=0.0))
    opt = torch.optim.AdamW(groups)
    bs, epochs = cfg["bs"], (1 if smoke else cfg["epochs"])
    total = math.ceil(len(tr) / bs) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=CUDA)
    counts = np.bincount(y[tr], minlength=3)
    cw = torch.tensor(len(tr) / (3 * counts), dtype=torch.float32, device=DEVICE)   # 'balanced' weights
    hist, t0 = [], time.time()
    for ep in range(epochs):
        model.train(); perm = rng.permutation(len(tr)); run_loss = 0.0; nb = 0
        for s in range(0, len(tr), bs):
            pos = perm[s:s + bs]
            inp, att = make_batch(tok_train, tr[pos], pad_id)
            yy = torch.tensor(y[tr[pos]], dtype=torch.long, device=DEVICE)
            pr = torch.tensor(p_tr[pos], dtype=torch.float32, device=DEVICE) if use_prior else None
            with torch.autocast("cuda", dtype=torch.float16, enabled=CUDA):
                logits = model(inp, att, pr)
            loss = F.cross_entropy(logits.float(), yy, weight=cw)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += float(loss); nb += 1
        vp = predict(model, tok_train, va, p_va, pad_id)
        f1 = mf1_proba(y[va], vp); hist.append(f1)
        print(f"  [{cfg['name']}] fold {k} epoch {ep+1}/{epochs} loss={run_loss/nb:.4f} val macroF1={f1:.4f} "
              f"({time.time()-t0:.0f}s elapsed)", flush=True)
    # FIXED schedule: final epoch is used (no best-epoch picking -> no optimistic OOF bias)
    tp = predict(model, tok_test, np.arange(len(test_ids)), p_te, pad_id)
    del model, backbone, opt; gc.collect(); torch.cuda.empty_cache()
    return va, vp, tp, hist

def run_transformer(cfg, smoke=False):
    name = cfg["name"]
    final = RUN_DIR / f"{name}_final.npz"
    if final.exists() and not smoke:
        z = np.load(final); print(f"[{name}] already finished -> OOF macroF1 = {mf1_proba(y, z['oof']):.6f}"); return z["oof"], z["test"]
    tok = get_tokenizer(cfg["model"]); pad_id = tok.pad_token_id
    norm = (lambda s: s)
    if cfg.get("csebuet_norm"):
        try:
            from normalizer import normalize
            norm = lambda s: normalize(s)
        except Exception as e:
            print("WARN csebuetnlp normalizer unavailable ->", e)
    enc = lambda texts: tok([norm(t) for t in texts], truncation=True, max_length=cfg["max_len"])["input_ids"]
    tok_train, tok_test = enc(list(clean_texts)), enc(list(test_clean))
    oof = np.zeros((len(y), 3), dtype=np.float32); tests = []
    folds = [0] if smoke else range(N_SPLITS)
    for k in folds:
        fp = RUN_DIR / f"{name}_fold{k}.npz"
        if fp.exists() and not smoke:
            z = np.load(fp); va, vp, tp = z["va"], z["vp"], z["tp"]; print(f"  [{name}] fold {k} loaded from disk")
        else:
            va, vp, tp, hist = run_fold(cfg, k, tok_train, tok_test, pad_id, smoke=smoke)
            if not smoke: np.savez(fp, va=va, vp=vp, tp=tp, hist=np.array(hist))
        oof[va] = vp; tests.append(tp)
    if smoke:
        print("SMOKE TEST OK - pipeline runs end-to-end."); return None, None
    test_avg = np.mean(tests, axis=0)
    np.savez(final, oof=oof, test=test_avg)
    print(f"[{name}] DONE  OOF macroF1 = {mf1_proba(y, oof):.6f}  per-class {per_class(y, oof)}")
    return oof, test_avg

# ------- what to run (edit freely). Each run ~45-60 min on a T4. Finished folds are cached, so re-running resumes. -------
RUNS = [
    dict(name="xlmr_base",        model="xlm-roberta-base", use_prior=False, epochs=3, lr=2e-5, bs=32, max_len=128),
    dict(name="xlmr_base_idhead", model="xlm-roberta-base", use_prior=True,  epochs=3, lr=2e-5, bs=32, max_len=128),
    # optional extra diversity (uncomment):
    # dict(name="banglabert", model="csebuetnlp/banglabert", use_prior=False, epochs=3, lr=3e-5, bs=32, max_len=128, csebuet_norm=True),
    # dict(name="muril_base", model="google/muril-base-cased", use_prior=False, epochs=3, lr=3e-5, bs=32, max_len=128),
]


In [ ]:
# ============ CELL 3b: SMOKE TEST (~2 min). Run this ONCE before the long runs. ============
_ = run_transformer(dict(RUNS[1]), smoke=True)   # id-head variant exercises the most code


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [xlmr_base_idhead] fold 0 epoch 1/1 loss=1.0870 val macroF1=0.4351 (11s elapsed)
SMOKE TEST OK - pipeline runs end-to-end.


In [ ]:
# ============ CELL 3c: RUN THE TRANSFORMERS (long). Safe to re-run after a disconnect - resumes per fold. ============
transformer_out = {}
for cfg in RUNS:
    transformer_out[cfg["name"]] = run_transformer(cfg)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [xlmr_base] fold 0 epoch 1/3 loss=1.0090 val macroF1=0.5099 (256s elapsed)
  [xlmr_base] fold 0 epoch 2/3 loss=0.8776 val macroF1=0.5298 (516s elapsed)
  [xlmr_base] fold 0 epoch 3/3 loss=0.7913 val macroF1=0.5401 (775s elapsed)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [xlmr_base_idhead] fold 3 epoch 1/3 loss=0.8094 val macroF1=0.6162 (259s elapsed)
  [xlmr_base_idhead] fold 3 epoch 2/3 loss=0.7121 val macroF1=0.6161 (518s elapsed)
  [xlmr_base_idhead] fold 3 epoch 3/3 loss=0.6481 val macroF1=0.6348 (776s elapsed)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [xlmr_base_idhead] fold 4 epoch 1/3 loss=0.8123 val macroF1=0.6456 (259s elapsed)
  [xlmr_base_idhead] fold 4 epoch 2/3 loss=0.7082 val macroF1=0.6663 (518s elapsed)
  [xlmr_base_idhead] fold 4 epoch 3/3 loss=0.6429 val macroF1=0.6503 (778s elapsed)
[xlmr_base_idhead] DONE  OOF macroF1 = 0.647554  per-class [0.76, 0.608, 0.5747]


In [ ]:
# ============ CELL 4: FROZEN MULTILINGUAL-E5 EMBEDDINGS + LR HEAD (cheap, ~10 min total) ============
from sklearn.preprocessing import StandardScaler
EMB_MODELS = ["intfloat/multilingual-e5-small", "intfloat/multilingual-e5-base"]

@torch.no_grad()
def encode(model_name, texts, bs=128, max_len=128):
    tok = get_tokenizer(model_name); mdl = get_backbone(model_name).to(DEVICE).eval()
    if CUDA: mdl = mdl.half()
    texts = ["query: " + t for t in texts]                     # e5 convention
    order = np.argsort([len(t) for t in texts]); out = np.zeros((len(texts), mdl.config.hidden_size), dtype=np.float32)
    for s in range(0, len(texts), bs):
        pos = order[s:s + bs]
        b = tok([texts[i] for i in pos], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
        h = mdl(**b).last_hidden_state.float(); m = b["attention_mask"].unsqueeze(-1).float()
        e = (h * m).sum(1) / m.sum(1).clamp_min(1)             # mean pooling
        out[pos] = F.normalize(e, dim=-1).cpu().numpy()
    del mdl; gc.collect(); torch.cuda.empty_cache()
    return out

emb_out = {}
for mname in EMB_MODELS:
    tag = "emb_" + mname.split("/")[-1].replace("multilingual-", "")
    fin = RUN_DIR / f"{tag}_final.npz"
    if fin.exists():
        z = np.load(fin); emb_out[tag] = (z["oof"], z["test"]); print(f"[{tag}] loaded  OOF macroF1 = {mf1_proba(y, z['oof']):.6f}"); continue
    t0 = time.time()
    E = encode(mname, list(clean_texts) + list(test_clean)); Etr, Ete = E[:len(y)], E[len(y):]
    oof = np.zeros((len(y), 3)); tests = []
    for k in range(N_SPLITS):
        tr, va = fold_split(k)
        sc = StandardScaler().fit(Etr[tr])
        clf = LogisticRegression(solver="lbfgs", C=4.0, class_weight="balanced", max_iter=2000, random_state=SEED).fit(sc.transform(Etr[tr]), y[tr])
        oof[va] = clf.predict_proba(sc.transform(Etr[va])); tests.append(clf.predict_proba(sc.transform(Ete)))
    test_avg = np.mean(tests, axis=0); np.savez(fin, oof=oof, test=test_avg); emb_out[tag] = (oof, test_avg)
    print(f"[{tag}] OOF macroF1 = {mf1_proba(y, oof):.6f} per-class {per_class(y, oof)} ({time.time()-t0:.0f}s)")


config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[emb_e5-base] OOF macroF1 = 0.521535 per-class [0.6512, 0.4698, 0.4436] (197s)


In [ ]:
# ============ CELL 5: HONEST CROSS-FITTED BLEND + REPORT (no submission file is written) ============
def compositions(n, T):
    if n == 1: yield (T,); return
    for i in range(T + 1):
        for rest in compositions(n - 1, T - i): yield (i,) + rest

def search_weights(S, rows, step):
    """S: (n_sources, N, 3). Best simplex weights maximising macro-F1 on `rows` only."""
    n = S.shape[0]; T = int(round(1 / step)); Sr = S[:, rows]; yr = y[rows]
    best_w, best_f = None, -1
    for c in compositions(n, T):
        w = np.array(c) / T
        f = mf1(yr, np.tensordot(w, Sr, 1).argmax(1))
        if f > best_f: best_f, best_w = f, w
    return best_w, best_f

def honest_blend(names, sources, step=None):
    S = np.stack([sources[n][0] for n in names]); n = len(names)
    step = step or (0.05 if n <= 4 else 0.1)
    pooled = np.zeros((len(y), 3)); fold_f, ws = [], []
    for k in range(N_SPLITS):
        tr, va = fold_split(k)
        w, _ = search_weights(S, tr, step)                    # tuned on the OTHER 4 folds only
        pooled[va] = np.tensordot(w, S[:, va], 1); fold_f.append(mf1_proba(y[va], pooled[va])); ws.append(w)
    return dict(names=names, score=mf1_proba(y, pooled), per_class=per_class(y, pooled), fold_scores=fold_f,
                fold_std=float(np.std(fold_f)), fold_weights=[list(map(float, w)) for w in ws], pooled=pooled, step=step)

# ---- collect every finished source: name -> (oof, test) ----
SOURCES = {"text": (oof_text, test_text), "id": (oof_id, test_id)}
for k_, v_ in globals().get("emb_out", {}).items(): SOURCES[k_] = v_
for cfg in RUNS:
    fin = RUN_DIR / f"{cfg['name']}_final.npz"
    if fin.exists():
        z = np.load(fin); SOURCES[cfg["name"]] = (z["oof"], z["test"])
print("sources:", list(SOURCES))
REF = 0.639579
print("\n--- single sources (OOF macro-F1) ---")
for n_, (o_, _t) in SOURCES.items(): print(f"{n_:22s} {mf1_proba(y, o_):.6f}   per-class {per_class(y, o_)}")

print("\n--- honest cross-fitted blends (weights tuned on 4 folds, scored on the 5th) ---")
base = honest_blend(["text", "id"], SOURCES)
print(f"{'text+id (baseline)':44s} {base['score']:.6f}  (locked ref {REF})")
results = {"text+id": base}
extras = [n for n in SOURCES if n not in ("text", "id")]
for n_ in extras:                                              # ablation: each new source added to text+id
    r = honest_blend(["text", "id", n_], SOURCES); results[f"text+id+{n_}"] = r
    print(f"{'text+id+' + n_:44s} {r['score']:.6f}  delta {r['score']-REF:+.6f}  fold_std {r['fold_std']:.4f}")
if len(extras) >= 2:
    allr = honest_blend(["text", "id"] + extras, SOURCES); results["ALL"] = allr
    print(f"{'ALL sources':44s} {allr['score']:.6f}  delta {allr['score']-REF:+.6f}  fold_std {allr['fold_std']:.4f}")

best_key = max(results, key=lambda k_: results[k_]["score"]); best = results[best_key]
band = ("REJECTED" if best["score"] < REF else "MARGINAL" if best["score"] < .645 else "PROMISING" if best["score"] < .655 else
        "STRONG" if best["score"] < .665 else "MAJOR" if best["score"] < .68 else "NUMBER-1 LEVEL")
print(f"\n=== BEST honest blend: {best_key}  ->  {best['score']:.6f}   ({band})   delta vs 0.639579 = {best['score']-REF:+.6f}")
print("per-class F1:", best["per_class"], "| fold scores:", [round(v, 4) for v in best["fold_scores"]], "| fold std:", round(best["fold_std"], 4))
print("weights chosen per fold (order = %s):" % best["names"]); [print("  ", w) for w in best["fold_weights"]]

# ---- final weights on ALL OOF (for test-time use) + candidate test probabilities (NOT a submission) ----
names = best["names"]; S_all = np.stack([SOURCES[n][0] for n in names])
w_final, f_opt = search_weights(S_all, np.arange(len(y)), best["step"])
test_blend = np.tensordot(w_final, np.stack([SOURCES[n][1] for n in names]), 1)
np.save(WORK / "CANDIDATE_NOT_FOR_SUBMISSION_test_probs.npy", test_blend)
np.save(WORK / "CANDIDATE_oof_blend_probs.npy", best["pooled"])
summary = {k_: {kk: v for kk, v in r.items() if kk != "pooled"} for k_, r in results.items()}
summary["best"] = best_key; summary["final_weights"] = dict(zip(names, map(float, w_final)))
summary["single_sources"] = {n_: mf1_proba(y, o_) for n_, (o_, _t) in SOURCES.items()}
json.dump(summary, open(WORK / "results_summary.json", "w"), indent=2)
print("\nfinal weights (fit on all OOF):", dict(zip(names, np.round(w_final, 3))), f"(same-OOF optimistic score {f_opt:.6f})")
print("test predicted class share:", np.round(np.bincount(test_blend.argmax(1), minlength=3) / len(test_blend), 4),
      "| train share:", np.round(np.bincount(y) / len(y), 4))
print("\nNO submission file was created. Send results_summary.json / the printed table back for review.")


sources: ['text', 'id', 'emb_e5-small', 'emb_e5-base', 'xlmr_base', 'xlmr_base_idhead']

--- single sources (OOF macro-F1) ---
text                   0.515571   per-class [0.64, 0.4959, 0.4108]
id                     0.613631   per-class [0.733, 0.6025, 0.5054]
emb_e5-small           0.498531   per-class [0.6325, 0.447, 0.4161]
emb_e5-base            0.521535   per-class [0.6512, 0.4698, 0.4436]
xlmr_base              0.541633   per-class [0.6703, 0.4801, 0.4745]
xlmr_base_idhead       0.647554   per-class [0.76, 0.608, 0.5747]

--- honest cross-fitted blends (weights tuned on 4 folds, scored on the 5th) ---
text+id (baseline)                           0.640821  (locked ref 0.639579)
text+id+emb_e5-small                         0.659611  delta +0.020032  fold_std 0.0048
text+id+emb_e5-base                          0.665812  delta +0.026233  fold_std 0.0052
text+id+xlmr_base                            0.668240  delta +0.028661  fold_std 0.0044
text+id+xlmr_base_idhead                   

In [ ]:
# ============ CELL 6: ZIP small artifacts (OOF/test probs + summary) for download ============
import shutil
from google.colab import files
shutil.make_archive("/content/uap_results", "zip", WORK_DIR)
files.download("/content/uap_results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>